# Kannada-MNIST

## PCA + K-means overclustering with mixed learning data

Fit **PCA retaining at least 90% of learning variance** on **48,000 TRAIN + 8,192 Dig-MNIST learning images**, then fit **K=30** centers in the PCA space. Assign each center a digit from its learning members' majority label. Keep the 12,000 TRAIN and 2,048 Dig validation images excluded from both PCA fitting and clustering/mapping.

- [Competition](https://www.kaggle.com/competitions/Kannada-MNIST)
- [PCA documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Consolidated experiment report](report.ipynb)

Normalize the original 784 pixels to [0,1]. PCA centers these features and uses full SVD to select the smallest leading-component count whose cumulative explained variance reaches 90%. The component count is determined from the 56,192 learning images only, not validation performance. **Whitening is disabled**, so this is a truncated projection without component variance rescaling. PCA receives images only, not labels. For an image x:

$$z=(x-\mu)W_d,\qquad \hat{x}=\mu+zW_d^{T}.$$

The K-means objective is the sum of squared distances **in the automatically selected d-dimensional PCA coordinates**. It excludes discarded directions; it is not the full 784-pixel squared distance. Labels are used only afterward for learning-only majority mapping. Cluster IDs need not correspond one-to-one to digits.

Use 50 k-means++ initializations with the same seed list, K=30, at most 300 iterations per run, tolerance 0.0001, and minimum-learning-inertia selection. Each combined learning image has equal weight. No alignment, augmentation, source reweighting, or cluster-size constraint is used. CPU execution is intentional. No upload or competition submission is performed.

PCA and K-means are saved together under a separate artifact directory. Reuse requires matching inputs, PCA settings, clustering settings, seed list, and software versions. Cross-experiment comparisons belong in `report.ipynb`; this notebook describes its own PCA representation and selected model.


In [ ]:
import hashlib
import json
import sys
import time
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    confusion_matrix,
    normalized_mutual_info_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits
from IPython.display import display

SEED = 42
print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("Execution: CPU (scikit-learn KMeans)")
environment = {
    name: version(name)
    for name in [
        "numpy",
        "pandas",
        "scikit-learn",
        "plotly",
        "kagglehub",
        "joblib",
        "threadpoolctl",
    ]
}
display(pd.Series(environment, name="version").to_frame())
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"


## Data preparation

Use KaggleHub for missing CSVs. Reuse local files, keep original row order, and preserve sample-submission IDs.

In [ ]:
import kagglehub
from dotenv import load_dotenv

COMPETITION = "Kannada-MNIST"
repo_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "competitions" / COMPETITION).is_dir()
    ),
    None,
)
project_dir = (
    repo_dir / "competitions" / COMPETITION
    if repo_dir is not None
    else Path.cwd() if Path.cwd().name == COMPETITION else Path.cwd() / COMPETITION
)
output_dir = project_dir / "output"

if repo_dir is not None:
    load_dotenv(repo_dir / ".env", override=False)

required_files = ("train.csv", "test.csv", "sample_submission.csv", "Dig-MNIST.csv")
data_dir = project_dir / "data"
if not all((data_dir / name).is_file() for name in required_files):
    data_dir = Path(
        kagglehub.competition_download(
            COMPETITION,
            output_dir=str(data_dir.resolve()),
        )
    )

missing = [name for name in required_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing competition files in {data_dir}: {missing}")

print("Competition CSVs ready:", ", ".join(required_files))

output_dir = project_dir / "output" / "experiment-kmeans-pca-overclustered-mixed"
output_dir.mkdir(parents=True, exist_ok=True)

print("Artifacts are stored under the competition-local output directory.")


In [ ]:
train = pd.read_csv(data_dir / "train.csv", dtype=np.int16)
dig_mnist = pd.read_csv(data_dir / "Dig-MNIST.csv", dtype=np.int16)
submission_template = pd.read_csv(data_dir / "sample_submission.csv")
test = pd.read_csv(data_dir / "test.csv")

pixels = [f"pixel{i}" for i in range(784)]
assert train.columns.tolist() == ["label", *pixels]
assert dig_mnist.columns.tolist() == ["label", *pixels]
assert submission_template.columns.tolist() == ["id", "label"]
assert test.columns.tolist() == ["id", *pixels]
assert len(test) == len(submission_template)
assert test["id"].is_unique
assert test["id"].equals(submission_template["id"])

for name, frame in [("train", train), ("Dig-MNIST", dig_mnist), ("test", test)]:
    assert not frame.isna().any().any(), f"Missing values in {name}"
    assert frame[pixels].min().min() >= 0 and frame[pixels].max().max() <= 255
    if "label" in frame:
        assert frame["label"].between(0, 9).all()

display(
    pd.DataFrame(
        [
            {"file": "train.csv", "rows": len(train), "columns": len(train.columns)},
            {
                "file": "Dig-MNIST.csv",
                "rows": len(dig_mnist),
                "columns": len(dig_mnist.columns),
            },
            {"file": "test.csv", "rows": len(test), "columns": len(test.columns)},
            {
                "file": "sample_submission.csv",
                "rows": len(submission_template),
                "columns": len(submission_template.columns),
            },
        ]
    )
)
display(train.head())
display(submission_template.head())


## Fixed evaluation protocol

Use the existing independently stratified 80/20 splits with seed 42 and original CSV row order. Fit PCA, centers, and majority mapping on **TRAIN learning 48,000 + Dig learning 8,192** rows. The **12,000 TRAIN validation** and **2,048 Dig-MNIST validation** images remain excluded from PCA fitting, center fitting, and mapping. No competition test image is used for training and no held-out label is used to choose cluster labels or tune K.

Store both CSV hashes and all row positions in `split_manifest.json`. The split settings are defined once below; no external split configuration is loaded.


In [ ]:
# Define the split once here; use the same settings and CSV row order in other notebooks.
split_config = {
    "seed": 42,
    "validation_fraction": 0.2,
    "stratify_column": "label",
    "row_order": "original CSV order",
}
all_indices = np.arange(len(train))
train_indices, valid_indices = train_test_split(
    all_indices,
    test_size=split_config["validation_fraction"],
    random_state=split_config["seed"],
    stratify=train[split_config["stratify_column"]].to_numpy(),
)
assert not np.intersect1d(train_indices, valid_indices).size
assert np.array_equal(
    np.sort(np.concatenate([train_indices, valid_indices])), all_indices
)
assert len(train_indices) == int(len(train) * 0.8)

all_dig_indices = np.arange(len(dig_mnist))
dig_train_indices, dig_valid_indices = train_test_split(
    all_dig_indices,
    test_size=split_config["validation_fraction"],
    random_state=split_config["seed"],
    stratify=dig_mnist[split_config["stratify_column"]].to_numpy(),
)
assert not np.intersect1d(dig_train_indices, dig_valid_indices).size
assert np.array_equal(
    np.sort(np.concatenate([dig_train_indices, dig_valid_indices])), all_dig_indices
)

dig_csv_sha256 = hashlib.sha256((data_dir / "Dig-MNIST.csv").read_bytes()).hexdigest()
train_csv_sha256 = hashlib.sha256((data_dir / "train.csv").read_bytes()).hexdigest()
split_manifest = {
    **split_config,
    "train_csv_sha256": train_csv_sha256,
    "dig_csv_sha256": dig_csv_sha256,
    "dig_training_rows": dig_train_indices.tolist(),
    "dig_validation_rows": dig_valid_indices.tolist(),
    "scikit_learn_version": version("scikit-learn"),
    "training_rows": train_indices.tolist(),
    "validation_rows": valid_indices.tolist(),
}
(output_dir / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2) + "\n"
)
split_counts = pd.DataFrame(
    {
        "TRAIN learning": train.iloc[train_indices]["label"]
        .value_counts()
        .sort_index(),
        "TRAIN validation": train.iloc[valid_indices]["label"]
        .value_counts()
        .sort_index(),
        "Dig learning": dig_mnist.iloc[dig_train_indices]["label"]
        .value_counts()
        .sort_index(),
        "Dig validation": dig_mnist.iloc[dig_valid_indices]["label"]
        .value_counts()
        .sort_index(),
    }
)
display(split_counts)
print("Training rows:", len(train_indices), "Validation rows:", len(valid_indices))
print("Split seed:", split_config["seed"])

print(
    "Dig learning rows:",
    len(dig_train_indices),
    "Dig validation rows:",
    len(dig_valid_indices),
)


## Fit centers and map clusters to digits

Generate 50 distinct run seeds from master seed 42. Each run uses one stochastic k-means++ initialization (`n_init=1`); select the smallest PCA-space learning inertia across all 50 runs, breaking exact ties by run order. Save every run seed, inertia, iteration count, and the selected run. Validation accuracy is computed only after selection; minimum inertia does not guarantee maximum digit accuracy. Learn the mapping from final nearest-center assignments of the combined TRAIN + Dig learning rows; ties deterministically choose the smallest digit. Training purity is a description of cluster composition, not validation accuracy or a calibrated probability. Runtime covers all initializations, whereas `n_iter_` describes only the selected run. K-means has no CNN epochs or cross-entropy training curves.

### Reuse a matching saved PCA and model

Before fitting, check the competition-local `model.joblib`. Reuse it only when its cache key matches the actual learning features, labels, original learning-row order, both source CSV hashes, preprocessing description, K-means settings, all 50 run seeds, NumPy/scikit-learn versions, and numerical thread setting. Missing, incompatible, or unreadable caches trigger a new search. Set `FORCE_REFIT = True` in the next cell to recompute all initializations deliberately.

A complete cache hit skips PCA and center fitting: recompute the learning-only label mapping, cluster diagnostics, validation metrics, and test predictions so the notebook reflects the current evaluation data. `fit_seconds_all_initializations` records the original training duration; `model_load_or_fit_seconds` and `cache_hit` describe this execution. Load only locally generated, trusted joblib artifacts.


In [ ]:
def pixel_features(frame):
    return np.ascontiguousarray(frame[pixels].to_numpy(dtype=np.float32) / 255.0)


# Preserve source order: TRAIN learning rows first, then Dig-MNIST learning rows.
x_train = np.concatenate(
    [
        pixel_features(train.iloc[train_indices]),
        pixel_features(dig_mnist.iloc[dig_train_indices]),
    ],
    axis=0,
)
y_train = np.concatenate(
    [
        train.iloc[train_indices]["label"].to_numpy(dtype=np.int64),
        dig_mnist.iloc[dig_train_indices]["label"].to_numpy(dtype=np.int64),
    ]
)
training_sources = {"TRAIN": len(train_indices), "Dig-MNIST": len(dig_train_indices)}
assert len(x_train) == len(y_train) == sum(training_sources.values())
assert not np.intersect1d(train_indices, valid_indices).size
assert not np.intersect1d(dig_train_indices, dig_valid_indices).size
print("Learning rows by source:", training_sources)
x_valid = pixel_features(train.iloc[valid_indices])
y_valid = train.iloc[valid_indices]["label"].to_numpy(dtype=np.int64)
x_dig_valid = pixel_features(dig_mnist.iloc[dig_valid_indices])
y_dig_valid = dig_mnist.iloc[dig_valid_indices]["label"].to_numpy(dtype=np.int64)

# PCA sees only the combined learning images; held-out rows are transformed later.
pca_config = {
    "n_components": 0.90,
    "svd_solver": "full",
    "whiten": False,
}
# Set True to ignore both the fitted PCA and K-means cache.
FORCE_REFIT = False
pca_cache_context = {
    "cache_schema": 1,
    "learning_pixels_sha256": hashlib.sha256(memoryview(x_train)).hexdigest(),
    "learning_shape": list(x_train.shape),
    "learning_dtype": str(x_train.dtype),
    "train_csv_sha256": train_csv_sha256,
    "dig_csv_sha256": dig_csv_sha256,
    "config": pca_config,
    "numpy_version": version("numpy"),
    "sklearn_version": version("scikit-learn"),
    "numerical_threads": 1,
}
pca_cache_key = hashlib.sha256(
    json.dumps(pca_cache_context, sort_keys=True).encode()
).hexdigest()
model_cache_path = output_dir / "model.joblib"
pca = None
pca_cache_hit = False
if model_cache_path.is_file() and not FORCE_REFIT:
    try:
        pipeline_cache = joblib.load(model_cache_path)
    except (OSError, EOFError, ValueError):
        print("Saved PCA could not be read; fitting a new transform.")
    else:
        if (
            isinstance(pipeline_cache, dict)
            and pipeline_cache.get("pca_cache_key") == pca_cache_key
            and {"pca", "pca_fit_seconds"}.issubset(pipeline_cache)
        ):
            pca = pipeline_cache["pca"]
            pca_fit_seconds = pipeline_cache["pca_fit_seconds"]
            pca_cache_hit = True
            print("Loaded matching saved PCA transform.")
if pca is None:
    pca = PCA(**pca_config)
    pca_started = time.perf_counter()
    with threadpool_limits(limits=1):
        pca.fit(x_train)
    pca_fit_seconds = time.perf_counter() - pca_started
# Normalize array layout so freshly fitted and reloaded PCA use the same projection path.
pca.components_ = np.ascontiguousarray(pca.components_)
pca.mean_ = np.ascontiguousarray(pca.mean_)
with threadpool_limits(limits=1):
    clustering_features_train = np.ascontiguousarray(
        pca.transform(x_train), dtype=np.float32
    )
assert clustering_features_train.shape == (len(x_train), pca.n_components_)
variance_target = pca_config["n_components"]
retained_learning_variance = float(
    np.sum(pca.explained_variance_ratio_, dtype=np.float64)
)
previous_learning_variance = float(
    np.sum(pca.explained_variance_ratio_[:-1], dtype=np.float64)
)
assert retained_learning_variance >= variance_target
assert previous_learning_variance < variance_target
pca_selection = {
    "target_explained_variance": variance_target,
    "selected_components": int(pca.n_components_),
    "retained_learning_variance": retained_learning_variance,
    "previous_component_cumulative_variance": previous_learning_variance,
    "fit_learning_samples": int(pca.n_samples_),
}
(output_dir / "pca_selection.json").write_text(
    json.dumps(pca_selection, indent=2) + "\n"
)
display(pd.Series(pca_selection, name="value").to_frame())
assert np.isfinite(clustering_features_train).all()
print("PCA dimensions:", x_train.shape[1], "→", clustering_features_train.shape[1])
print("Retained learning variance:", f"{pca.explained_variance_ratio_.sum():.2%}")

N_RESTARTS = 50
restart_seeds = np.random.RandomState(SEED).randint(
    0, np.iinfo(np.int32).max, size=N_RESTARTS
)
assert len(np.unique(restart_seeds)) == N_RESTARTS
kmeans_config = {
    "n_clusters": 30,
    "init": "k-means++",
    "n_init": 1,
    "max_iter": 300,
    "tol": 1e-4,
    "algorithm": "lloyd",
}
cache_context = {
    "cache_schema": 1,
    "feature_pipeline": "unwhitened PCA retaining at least 90% learning variance from 784 normalized pixels",
    "selected_pca_components": int(pca.n_components_),
    "pca_config": pca_config,
    "pca_cache_key": pca_cache_key,
    "training_shape": list(clustering_features_train.shape),
    "training_dtype": str(clustering_features_train.dtype),
    "training_features_sha256": hashlib.sha256(
        memoryview(clustering_features_train)
    ).hexdigest(),
    "training_labels_sha256": hashlib.sha256(memoryview(y_train)).hexdigest(),
    "learning_rows_sha256": hashlib.sha256(
        np.asarray(train_indices, dtype="<i8").tobytes()
    ).hexdigest(),
    "train_csv_sha256": train_csv_sha256,
    "dig_csv_sha256": dig_csv_sha256,
    "dig_learning_rows_sha256": hashlib.sha256(
        np.asarray(dig_train_indices, dtype="<i8").tobytes()
    ).hexdigest(),
    "training_sources": training_sources,
    "learning_row_order": "TRAIN learning rows, then Dig learning rows; split-index order within each source",
    "master_seed": SEED,
    "restart_seeds": restart_seeds.tolist(),
    "config": kmeans_config,
    "numpy_version": version("numpy"),
    "sklearn_version": version("scikit-learn"),
    "numerical_threads": 1,
}
training_cache_key = hashlib.sha256(
    json.dumps(cache_context, sort_keys=True).encode()
).hexdigest()
model_cache_path = output_dir / "model.joblib"
cache_started = time.perf_counter()
cached_search = None
if model_cache_path.is_file() and not FORCE_REFIT:
    try:
        candidate_cache = joblib.load(model_cache_path)
    except (OSError, EOFError, ValueError):
        print("Saved model could not be read; fitting a new model.")
    else:
        required = {"model", "restart_history", "fit_seconds"}
        if (
            isinstance(candidate_cache, dict)
            and candidate_cache.get("training_cache_key") == training_cache_key
            and required.issubset(candidate_cache)
        ):
            cached_search = candidate_cache
        else:
            print("Saved model does not match the current training inputs or settings.")

cache_hit = cached_search is not None
if cache_hit:
    model = cached_search["model"]
    restart_history = cached_search["restart_history"]
    fit_seconds = float(cached_search["fit_seconds"])
    best_restart = int(min(restart_history, key=lambda row: row["inertia"])["restart"])
    print("Loaded matching saved model; skipped all 50 initialization fits.")
else:
    restart_history = []
    model = None
    best_inertia = np.inf
    best_restart = None
    started = time.perf_counter()
    # Fit each stochastic initialization independently; use no validation data or labels.
    with threadpool_limits(limits=1):
        for restart, restart_seed in enumerate(restart_seeds, start=1):
            run_started = time.perf_counter()
            candidate = KMeans(**kmeans_config, random_state=int(restart_seed)).fit(
                clustering_features_train
            )
            restart_history.append(
                {
                    "restart": restart,
                    "seed": int(restart_seed),
                    "inertia": float(candidate.inertia_),
                    "iterations": int(candidate.n_iter_),
                    "seconds": time.perf_counter() - run_started,
                }
            )
            if candidate.inertia_ < best_inertia:
                model = candidate
                best_inertia = float(candidate.inertia_)
                best_restart = restart
            pd.DataFrame(restart_history).to_csv(
                output_dir / "restart_history.csv", index=False
            )
            if restart == 1 or restart % 10 == 0:
                print(
                    f"Restarts: {restart}/{N_RESTARTS}; best learning inertia: {best_inertia:,.2f}"
                )
    fit_seconds = time.perf_counter() - started
assert model is not None
restart_history_df = pd.DataFrame(restart_history)
restart_history_df["selected"] = restart_history_df["restart"] == best_restart
assert best_restart == int(
    restart_history_df.loc[restart_history_df["inertia"].idxmin(), "restart"]
)
assert float(model.inertia_) == restart_history_df["inertia"].min()
restart_history_df.to_csv(output_dir / "restart_history.csv", index=False)
display(restart_history_df.sort_values(["inertia", "restart"]).head(10))
fig = px.scatter(
    restart_history_df.assign(
        selection=np.where(
            restart_history_df["selected"], "Selected model", "Other initialization"
        )
    ),
    x="restart",
    y="inertia",
    color="selection",
    hover_data=["seed", "iterations"],
    labels={
        "restart": "Initialization run",
        "inertia": "Learning squared-distance sum",
        "selection": "Selection",
    },
    title="50 reproducible initializations: select minimum learning inertia",
)
fig.show()

# Use final nearest-center assignments consistently for mapping and inference.
with threadpool_limits(limits=1):
    learning_clusters = model.predict(clustering_features_train)
cluster_counts = np.zeros((model.n_clusters, 10), dtype=np.int64)
np.add.at(cluster_counts, (learning_clusters, y_train), 1)
assert (cluster_counts.sum(axis=1) > 0).all(), "Unexpected empty cluster"
cluster_to_digit = cluster_counts.argmax(axis=1)
cluster_purity = cluster_counts.max(axis=1) / cluster_counts.sum(axis=1)

mapping_df = pd.DataFrame(
    {
        "cluster": np.arange(model.n_clusters),
        "mapped_digit": cluster_to_digit,
        "kannada_digit": [chr(0x0CE6 + int(d)) for d in cluster_to_digit],
        "learning_samples": cluster_counts.sum(axis=1),
        "learning_purity": cluster_purity,
    }
)
fit_summary = {
    "training_samples": len(x_train),
    "features": clustering_features_train.shape[1],
    "original_features": x_train.shape[1],
    "clusters": model.n_clusters,
    "initializations": N_RESTARTS,
    "selected_restart": int(best_restart),
    "selected_seed": int(model.random_state),
    "best_run_iterations": int(model.n_iter_),
    "best_run_inertia": float(model.inertia_),
    "fit_seconds_all_initializations": fit_seconds,
    "cache_hit": cache_hit,
    "model_load_or_fit_seconds": time.perf_counter() - cache_started,
}
display(pd.Series(fit_summary, name="value").to_frame())
display(mapping_df.sort_values(["mapped_digit", "cluster"]))
print("Digits represented by majority mapping:", sorted(set(cluster_to_digit.tolist())))

cluster_order = np.lexsort((np.arange(model.n_clusters), cluster_to_digit))
kannada_digits = [chr(0x0CE6 + digit) for digit in range(10)]
cluster_labels = {
    int(c): f"{cluster_to_digit[c]} ({kannada_digits[cluster_to_digit[c]]}) · C{c}"
    for c in range(model.n_clusters)
}
cluster_names = [cluster_labels[int(c)] for c in cluster_order]
assert np.all(np.diff(cluster_to_digit[cluster_order]) >= 0)

model_cache = {
    "model": model,
    "pca": pca,
    "pca_cache_key": pca_cache_key,
    "pca_cache_context": pca_cache_context,
    "pca_fit_seconds": pca_fit_seconds,
    "cluster_to_digit": cluster_to_digit,
    "training_cache_key": training_cache_key,
    "cache_context": cache_context,
    "restart_history": restart_history_df.to_dict("records"),
    "fit_seconds": fit_seconds,
}
joblib.dump(model_cache, model_cache_path)


## PCA representation diagnostics

The cumulative explained-variance plot covers the **automatically retained components** and marks the 90% threshold and selected count. Full SVD determines the variance spectrum; the returned PCA object retains only the selected leading components. The cumulative variance with one fewer component must remain below 90%. Principal-component images use signed loadings with a shared zero-centered scale; they are variation directions, not digit prototypes.

Original/reconstructed pairs use one deterministic learning image per digit. The PCA reconstruction is displayed with a [0,1] color range; reported reconstruction MSE uses the unclipped values. These are learning examples for explanation, not a validation score or a choice of component count.


In [ ]:
component_numbers = np.arange(1, pca.n_components_ + 1)
variance_df = pd.DataFrame(
    {
        "component": component_numbers,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
    }
)
display(variance_df)
fig = px.line(
    variance_df,
    x="component",
    y="cumulative_explained_variance",
    markers=True,
    title=f"PCA variance target: {variance_target:.0%} retained with {pca.n_components_} components",
    labels={
        "component": "Retained components",
        "cumulative_explained_variance": "Explained learning variance",
    },
)
fig.add_hline(
    y=variance_target,
    line_dash="dash",
    annotation_text=f"Target: {variance_target:.0%}",
)
fig.add_vline(
    x=int(pca.n_components_),
    line_dash="dot",
    annotation_text=f"Selected: {pca.n_components_}",
)
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

shown_components = min(10, int(pca.n_components_))
component_rows = (shown_components + 4) // 5
fig = make_subplots(
    rows=component_rows,
    cols=5,
    subplot_titles=[
        f"PC {i+1}: {pca.explained_variance_ratio_[i]:.1%}"
        for i in range(shown_components)
    ],
)
loading_limit = float(np.abs(pca.components_[:shown_components]).max())
for index, component in enumerate(pca.components_[:shown_components]):
    fig.add_trace(
        go.Heatmap(z=component.reshape(28, 28), coloraxis="coloraxis"),
        row=index // 5 + 1,
        col=index % 5 + 1,
    )
    axis = "x" if index == 0 else f"x{index+1}"
    fig.update_yaxes(
        autorange="reversed",
        visible=False,
        scaleanchor=axis,
        scaleratio=1,
        row=index // 5 + 1,
        col=index % 5 + 1,
    )
fig.update_xaxes(visible=False, constrain="domain")
fig.update_layout(
    title="Leading PCA directions: signed pixel loadings",
    height=component_rows * 240 + 100,
    coloraxis=dict(
        colorscale="RdBu",
        cmin=-loading_limit,
        cmax=loading_limit,
        cmid=0,
        colorbar=dict(title="Loading"),
    ),
)
fig.show()

learning_examples = np.array([np.flatnonzero(y_train == d)[0] for d in range(10)])
with threadpool_limits(limits=1):
    reconstructed = pca.inverse_transform(clustering_features_train[learning_examples])
reconstruction_mse = np.mean((x_train[learning_examples] - reconstructed) ** 2, axis=1)
image_titles = []
for digit in range(10):
    image_titles.extend(
        [
            f"Original {digit} ({chr(0x0CE6+digit)})",
            f"PCA {digit} ({chr(0x0CE6+digit)})",
        ]
    )
fig = make_subplots(rows=2, cols=10, subplot_titles=image_titles, vertical_spacing=0.2)
for digit, position in enumerate(learning_examples):
    row = digit // 5 + 1
    for offset, image in enumerate([x_train[position], reconstructed[digit]]):
        column = (digit % 5) * 2 + offset + 1
        fig.add_trace(
            go.Heatmap(
                z=image.reshape(28, 28),
                colorscale="Gray",
                zmin=0,
                zmax=1,
                showscale=False,
            ),
            row=row,
            col=column,
        )
        axis_number = (row - 1) * 10 + column
        axis = "x" if axis_number == 1 else f"x{axis_number}"
        fig.update_yaxes(
            autorange="reversed",
            visible=False,
            scaleanchor=axis,
            scaleratio=1,
            row=row,
            col=column,
        )
fig.update_xaxes(visible=False, constrain="domain")
fig.update_layout(
    title=f"Learning examples: original and {pca.n_components_}-component reconstruction",
    height=560,
)
fig.show()
display(
    pd.DataFrame(
        {
            "digit": np.arange(10),
            "learning_position": learning_examples,
            "reconstruction_mse": reconstruction_mse,
        }
    )
)
variance_df.to_csv(output_dir / "pca_explained_variance.csv", index=False)


## Reconstructed centers and cluster composition

The model centers use the automatically selected number of PCA coordinates. Apply `pca.inverse_transform` to show each as a 28 × 28 reconstructed image in a five-column grid. Color limits [0,1] clip only the display; reconstructed coordinates can fall outside that range and are not clipped for distance computations. The gallery depicts projected prototypes, while the empirical raw-pixel mean and variance galleries describe the original assigned learning images.

Captions retain the numeric/Unicode mapped digit and original cluster ID. Composition counts include both learning sources and are sorted by mapped digit.


In [ ]:
def digit_gallery(images, titles, title, captions_below=False):
    count = len(images)
    columns = 5
    rows = (count + columns - 1) // columns
    fig = make_subplots(
        rows=rows,
        cols=columns,
        subplot_titles=None if captions_below else titles,
        vertical_spacing=0.30 if captions_below else min(0.15, 0.3 / rows),
    )
    for index, image in enumerate(images):
        fig.add_trace(
            go.Heatmap(
                z=image.reshape(28, 28),
                colorscale="Gray",
                zmin=0,
                zmax=1,
                showscale=False,
                hovertemplate="x=%{x}, y=%{y}<br>Intensity=%{z:.2f}<extra></extra>",
            ),
            row=index // columns + 1,
            col=index % columns + 1,
        )
    if captions_below:
        for index, caption in enumerate(titles):
            fig.add_annotation(
                text=caption,
                x=0.5,
                y=0,
                xref="x domain",
                yref="y domain",
                xanchor="center",
                yanchor="top",
                yshift=-10,
                showarrow=False,
                align="center",
                font=dict(size=12),
                row=index // columns + 1,
                col=index % columns + 1,
            )

    fig.update_xaxes(visible=False, constrain="domain")
    for index in range(rows * columns):
        axis = "x" if index == 0 else f"x{index + 1}"
        fig.update_yaxes(
            visible=False,
            autorange="reversed",
            scaleanchor=axis,
            scaleratio=1,
            row=index // columns + 1,
            col=index % columns + 1,
        )
    fig.update_layout(
        title=title,
        height=rows * (280 if captions_below else 230) + 100,
        margin=dict(t=80 if captions_below else 110, b=90 if captions_below else 40),
    )
    return fig


kannada_digits = [chr(0x0CE6 + digit) for digit in range(10)]
centroid_titles = [
    f"{cluster_labels[int(i)]}<br>Learning purity: {cluster_purity[i]:.1%}"
    for i in cluster_order
]
digit_gallery(
    pca.inverse_transform(model.cluster_centers_)[cluster_order],
    centroid_titles,
    "PCA-space centers reconstructed in pixel space",
).show()
fig = px.imshow(
    cluster_counts[cluster_order],
    x=[f"{i} ({kannada_digits[i]})" for i in range(10)],
    y=cluster_names,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Learning-set digit", "y": "Cluster", "color": "Samples"},
    title="Cluster composition: TRAIN + Dig learning rows",
)
fig.update_layout(height=max(450, model.n_clusters * 24 + 120))
fig.show()


## Cluster sizes and actual digit distributions

All diagnostics in this section use **TRAIN + Dig learning rows only**, assigned to their final nearest fitted center. TRAIN validation, Dig-MNIST validation, and competition test rows are excluded. Counts therefore sum to 56,192.

Show both absolute membership counts and each cluster's actual digit proportions. The proportions are $P(\mathrm{digit}\mid\mathrm{cluster})$, estimated from learning labels; they are not calibrated prediction probabilities. Label colors are consistent across plots. The mapped digit is the largest component, not necessarily the only digit present.

Digit labels are categorical, so an arithmetic mean or variance of the label numbers would not describe handwriting similarity. Use composition, majority purity, and label entropy instead. Entropy is zero for a single-digit cluster and at most $\log_2(10)$ bits for ten equally common digits.

All cluster plots and galleries are ordered by **mapped digit**, then original cluster ID to break ties. Display labels include both the numeric/Unicode mapped digit and original ID; sorting does not change assignments or the label mapping.

The individual-cluster distribution panel shows one bar for every true digit 0–9, including zero-count digits. Bar heights are learning-image counts on a shared scale; hover shows both the count and within-cluster percentage. Panels follow mapped digit, then original cluster ID. These true-label distributions explain how mixed each visual cluster is, even when its majority mapping suggests one digit.


In [ ]:
cluster_sizes = cluster_counts.sum(axis=1)
cluster_fractions = cluster_sizes / len(x_train)
digit_fractions = cluster_counts / cluster_sizes[:, None]
assert cluster_sizes.sum() == len(x_train)
assert np.allclose(digit_fractions.sum(axis=1), 1)

membership_df = mapping_df.assign(learning_fraction=cluster_fractions)
membership_df["cluster_label"] = membership_df["cluster"].map(cluster_labels)
membership_df = membership_df.sort_values(["mapped_digit", "cluster"])
display(
    membership_df.style.format(
        {"learning_purity": "{:.2%}", "learning_fraction": "{:.2%}"}
    )
)
fig = px.bar(
    membership_df,
    x="cluster_label",
    category_orders={"cluster_label": cluster_names},
    y="learning_samples",
    color="learning_purity",
    text="learning_samples",
    color_continuous_scale="Viridis",
    range_color=[0, 1],
    hover_data={
        "mapped_digit": True,
        "learning_fraction": ":.2%",
        "learning_purity": ":.2%",
    },
    labels={
        "cluster_label": "Mapped digit · cluster ID",
        "learning_samples": "Learning images",
        "learning_purity": "Majority purity",
    },
    title="Cluster sizes: counts and share of TRAIN learning rows",
)
fig.show()

composition_df = pd.DataFrame(
    [
        {
            "cluster": cluster_labels[int(c)],
            "digit": f"{d} ({kannada_digits[d]})",
            "count": int(cluster_counts[c, d]),
            "fraction": digit_fractions[c, d],
        }
        for c in cluster_order
        for d in range(10)
    ]
)
fig = px.bar(
    composition_df,
    x="cluster",
    y="fraction",
    color="digit",
    barmode="stack",
    category_orders={"cluster": cluster_names},
    hover_data={"count": True, "fraction": ":.2%"},
    labels={
        "cluster": "Cluster",
        "fraction": "Digit proportion within cluster",
        "digit": "True learning digit",
    },
    title="Actual digit composition inside each cluster",
)
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(legend_title_text="True digit; click to toggle")
fig.show()

fig = px.imshow(
    digit_fractions[cluster_order],
    x=[f"{d} ({kannada_digits[d]})" for d in range(10)],
    y=cluster_names,
    text_auto=".1%",
    aspect="auto",
    zmin=0,
    zmax=1,
    color_continuous_scale="Blues",
    labels={
        "x": "True learning digit",
        "y": "Cluster",
        "color": "Within-cluster fraction",
    },
    title="Row-normalized digit composition: each cluster sums to 100%",
)
fig.update_layout(height=max(450, model.n_clusters * 24 + 120))
fig.show()


columns = 5
rows = (model.n_clusters + columns - 1) // columns
fig = make_subplots(
    rows=rows,
    cols=columns,
    shared_yaxes=True,
    subplot_titles=cluster_names,
    vertical_spacing=min(0.18, 0.36 / rows),
)
for position, cluster in enumerate(cluster_order):
    fig.add_trace(
        go.Bar(
            x=[str(d) for d in range(10)],
            y=cluster_counts[cluster],
            marker_color=px.colors.qualitative.Plotly,
            customdata=digit_fractions[cluster],
            hovertemplate="True digit: %{x}<br>Learning images: %{y:,}<br>Within cluster: %{customdata:.2%}<extra></extra>",
            showlegend=False,
        ),
        row=position // columns + 1,
        col=position % columns + 1,
    )
fig.update_xaxes(
    title_text="True digit",
    type="category",
    categoryorder="array",
    categoryarray=[str(d) for d in range(10)],
    tickmode="array",
    tickvals=[str(d) for d in range(10)],
    tickfont=dict(size=10),
)
fig.update_yaxes(range=[0, float(cluster_counts.max()) * 1.1], rangemode="tozero")
fig.update_yaxes(title_text="Learning images", col=1)
fig.update_layout(
    title="True digit distribution in each learning cluster",
    height=rows * 280 + 100,
    margin=dict(t=90, b=55),
    bargap=0.12,
)
fig.show()


## Pixel means, pixel variances, and cluster spread

For each cluster, compute an empirical **28 × 28 mean image** and **28 × 28 population variance image** over its assigned learning images:

$$\mu_{c,p}=\frac{1}{n_c}\sum_{i\in c}x_{i,p},\qquad
v_{c,p}=\frac{1}{n_c}\sum_{i\in c}(x_{i,p}-\mu_{c,p})^2.$$

Inputs are intensities in [0,1], and population variance uses `ddof=0`. The variance image shows where members disagree at the **same pixel position**. All clusters share one color scale within a gallery; do not rescale each cluster independently. The reconstructed fitted centers are constrained to the retained PCA subspace. Raw-pixel empirical means and variances below are computed directly from assigned original images, so they describe appearance and variability without that reconstruction constraint.

The statistics table records mean intensity across all 784 pixels (including background), mean within-pixel variance across images, and label entropy. The empirical mean and variance images below provide the main visual explanation of cluster appearance and variation. A high intensity mean can reflect thicker/more extensive strokes; it is not a quality score. Squared radii use selected-dimensional PCA-coordinate distances to the fitted center; they are distinct from raw-pixel variance. These statistics describe the learning clusters and do not estimate held-out performance.

In [ ]:
cluster_pixel_means = []
cluster_pixel_variances = []
cluster_pca_means = []
cluster_pca_variances = []
cluster_statistics = []
learning_squared_distance = np.empty(len(x_train), dtype=np.float64)
for cluster in range(model.n_clusters):
    mask = learning_clusters == cluster
    members = x_train[mask]
    mean_image = members.mean(axis=0, dtype=np.float64)
    variance_image = members.var(axis=0, dtype=np.float64, ddof=0)
    pca_members = clustering_features_train[mask]
    cluster_pca_means.append(pca_members.mean(axis=0, dtype=np.float64))
    cluster_pca_variances.append(pca_members.var(axis=0, dtype=np.float64, ddof=0))
    squared_radius = np.sum(
        (pca_members - model.cluster_centers_[cluster]) ** 2, axis=1, dtype=np.float64
    )
    learning_squared_distance[mask] = squared_radius
    fractions = digit_fractions[cluster]
    positive = fractions[fractions > 0]
    cluster_pixel_means.append(mean_image)
    cluster_pixel_variances.append(variance_image)
    cluster_statistics.append(
        {
            "cluster": cluster,
            "samples": len(members),
            "mapped_digit": int(cluster_to_digit[cluster]),
            "majority_purity": cluster_purity[cluster],
            "label_entropy_bits": float(-(positive * np.log2(positive)).sum()),
            "mean_intensity": float(mean_image.mean()),
            "mean_within_pixel_variance": float(variance_image.mean()),
            "mean_squared_radius": float(squared_radius.mean()),
            "variance_squared_radius": float(squared_radius.var(ddof=0)),
            "median_squared_radius": float(np.median(squared_radius)),
            "q25_squared_radius": float(np.quantile(squared_radius, 0.25)),
            "q75_squared_radius": float(np.quantile(squared_radius, 0.75)),
        }
    )
cluster_pixel_means = np.stack(cluster_pixel_means)
cluster_pixel_variances = np.stack(cluster_pixel_variances)
cluster_pca_means = np.stack(cluster_pca_means)
cluster_pca_variances = np.stack(cluster_pca_variances)
cluster_statistics_df = pd.DataFrame(cluster_statistics)
assert cluster_pixel_means.shape == (model.n_clusters, 784)
assert cluster_pixel_variances.shape == (model.n_clusters, 784)
assert np.all(cluster_pixel_variances >= 0)
assert np.all(cluster_pixel_variances <= 0.25 + 1e-6)
assert np.isfinite(learning_squared_distance).all()
display(
    cluster_statistics_df.sort_values(["mapped_digit", "cluster"]).style.format(
        precision=4
    )
)


In [ ]:
def cluster_image_gallery(values, title, colorscale, maximum, colorbar_title):
    columns = 5
    rows = (model.n_clusters + columns - 1) // columns
    fig = make_subplots(
        rows=rows,
        cols=columns,
        vertical_spacing=min(0.14, 0.28 / rows),
        subplot_titles=[cluster_labels[int(i)] for i in cluster_order],
    )
    for position, cluster in enumerate(cluster_order):
        image = values[cluster]
        row, col = position // columns + 1, position % columns + 1
        fig.add_trace(
            go.Heatmap(
                z=image.reshape(28, 28),
                coloraxis="coloraxis",
                hovertemplate="x=%{x}, y=%{y}<br>Value=%{z:.4f}<extra></extra>",
            ),
            row=row,
            col=col,
        )
        axis = "x" if position == 0 else f"x{position + 1}"
        fig.update_yaxes(
            autorange="reversed",
            visible=False,
            scaleanchor=axis,
            scaleratio=1,
            row=row,
            col=col,
        )
    fig.update_xaxes(visible=False, constrain="domain")
    fig.update_layout(
        title=title,
        height=rows * 240 + 100,
        margin=dict(t=100),
        coloraxis=dict(
            cmin=0,
            cmax=maximum,
            colorscale=colorscale,
            colorbar=dict(title=colorbar_title),
        ),
    )
    return fig


cluster_image_gallery(
    cluster_pixel_means,
    "Empirical mean image of each learning cluster",
    "Gray",
    1.0,
    "Mean intensity",
).show()
variance_max = max(float(cluster_pixel_variances.max()), np.finfo(float).eps)
cluster_image_gallery(
    cluster_pixel_variances,
    "Per-pixel population variance: shared scale across clusters",
    "Magma",
    variance_max,
    "Pixel variance",
).show()


## Distances to fitted centers

A squared radius is the sum of squared differences across the selected PCA coordinates to the sample's assigned fitted center. The box plots and histograms use **all learning images**. Boxes summarize quartiles and whiskers; individual outliers are omitted from the drawing to reduce clutter, not removed from the statistics. Histograms share the same bins and are normalized within each cluster for shape comparison; membership sizes are reported separately above.

Large distances indicate unusual images relative to a fitted center. They are not probabilities, and a small distance does not prove the mapped digit is correct.

Read the distance histogram as follows:

- **Horizontal axis:** squared PCA distance to the assigned fitted center; it is not the digit label or a 2D position in the image.
- **Vertical axis:** the fraction of that cluster's learning images falling in a distance bin. A value of 10% means one tenth of the cluster's members fall in that bin.
- A distribution further to the right indicates larger differences in the retained PCA representation from its fitted center. A wider distribution indicates more variation in those distances.
- The box plot above summarizes the same distances with medians and quartiles. Neither graph measures classification accuracy or the probability of an error.


Whitening is disabled. Discarded principal directions contribute nothing to these distances. The displayed 28 × 28 center images are reconstructions; distances are always evaluated before reconstruction.

In [ ]:
distance_df = pd.DataFrame(
    {
        "cluster": [cluster_labels[int(c)] for c in learning_clusters],
        "squared_distance": learning_squared_distance,
    }
)
fig = px.box(
    distance_df,
    x="cluster",
    y="squared_distance",
    points=False,
    category_orders={"cluster": cluster_names},
    labels={
        "cluster": "Cluster",
        "squared_distance": "Squared PCA distance to fitted center",
    },
    title="Within-cluster spread: all mixed learning images",
)
fig.show()

bin_edges = np.linspace(
    0, float(learning_squared_distance.max()) + np.finfo(float).eps, 41
)
histogram_rows = []
for cluster in cluster_order:
    counts, _ = np.histogram(
        learning_squared_distance[learning_clusters == cluster], bins=bin_edges
    )
    for index, count in enumerate(counts):
        histogram_rows.append(
            {
                "cluster": cluster_labels[int(cluster)],
                "bin_center": (bin_edges[index] + bin_edges[index + 1]) / 2,
                "lower": bin_edges[index],
                "upper": bin_edges[index + 1],
                "count": int(count),
                "fraction": count / cluster_sizes[cluster],
            }
        )
histogram_df = pd.DataFrame(histogram_rows)
assert np.allclose(histogram_df.groupby("cluster")["fraction"].sum(), 1)
fig = px.line(
    histogram_df,
    x="bin_center",
    y="fraction",
    color="cluster",
    facet_col="cluster",
    facet_col_wrap=5,
    category_orders={"cluster": cluster_names},
    markers=True,
    hover_data={"lower": ":.2f", "upper": ":.2f", "count": True},
    labels={
        "bin_center": "Squared PCA distance",
        "fraction": "Fraction of cluster images",
    },
    title="Distance distributions: shared bins, normalized per cluster",
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(
    height=max(650, ((model.n_clusters + 4) // 5) * 240 + 120), showlegend=False
)
fig.show()

cluster_statistics_df.to_csv(output_dir / "cluster_statistics.csv", index=False)
composition_df.to_csv(output_dir / "cluster_digit_distribution.csv", index=False)
histogram_df.to_csv(output_dir / "cluster_distance_histogram.csv", index=False)
np.savez_compressed(
    output_dir / "cluster_pixel_statistics.npz",
    means=cluster_pixel_means,
    variances=cluster_pixel_variances,
    pca_means=cluster_pca_means,
    pca_variances=cluster_pca_variances,
)


## Held-out evaluation

Apply the fixed training-only mapping unchanged to both validation partitions. Report sample accuracy and macro precision/recall/F1, plus adjusted Rand index and normalized mutual information between true digits and cluster assignments. The latter two describe clustering agreement without requiring a cluster-to-digit permutation. They are evaluation metrics, not inputs to fitting or mapping.

In [ ]:
def predict_digits(features):
    with threadpool_limits(limits=1):
        clusters = model.predict(pca.transform(features))
    return cluster_to_digit[clusters], clusters


def evaluate(labels, predictions, clusters, name):
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=np.arange(10), zero_division=0
    )
    summary = {
        "dataset": name,
        "samples": len(labels),
        "accuracy": accuracy_score(labels, predictions),
        "precision": float(precision.mean()),
        "recall": float(recall.mean()),
        "f1": float(f1.mean()),
        "adjusted_rand": adjusted_rand_score(labels, clusters),
        "normalized_mutual_information": normalized_mutual_info_score(labels, clusters),
    }
    per_digit = pd.DataFrame(
        {
            "dataset": name,
            "digit": np.arange(10),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )
    return summary, per_digit


evaluation_sets = {
    "Validation": (x_valid, y_valid),
    "Dig-MNIST validation": (x_dig_valid, y_dig_valid),
}
summaries, per_digit_frames = [], []
evaluation_predictions = {}
for name, (features, labels) in evaluation_sets.items():
    predictions, clusters = predict_digits(features)
    evaluation_predictions[name] = (predictions, clusters)
    summary, per_digit = evaluate(labels, predictions, clusters, name)
    summaries.append(summary)
    per_digit_frames.append(per_digit)
    fig = px.imshow(
        confusion_matrix(labels, predictions, labels=np.arange(10)),
        x=[f"{i} ({kannada_digits[i]})" for i in range(10)],
        y=[f"{i} ({kannada_digits[i]})" for i in range(10)],
        text_auto=True,
        color_continuous_scale="Blues",
        labels={"x": "Predicted digit", "y": "True digit", "color": "Samples"},
        title=f"{name}: mapped K-means predictions",
    )
    fig.show()
scores_df = pd.DataFrame(summaries)
per_digit_df = pd.concat(per_digit_frames, ignore_index=True)
display(scores_df)
display(per_digit_df)


In [ ]:
comparison_rows = [{**row, "target": "Overall (macro)"} for row in summaries]
comparison_rows += [
    {**row, "target": f"Digit {row['digit']}"}
    for row in per_digit_df.to_dict("records")
]
comparison = pd.DataFrame(comparison_rows).melt(
    id_vars=["dataset", "target"],
    value_vars=["precision", "recall", "f1"],
    var_name="Metric",
    value_name="Score",
)
targets = ["Overall (macro)", *[f"Digit {i}" for i in range(10)]]
colors = px.colors.qualitative.Plotly + ["#666666"]
fig = px.bar(
    comparison,
    x="Metric",
    y="Score",
    color="target",
    facet_col="dataset",
    barmode="group",
    category_orders={"target": targets},
    color_discrete_map=dict(zip(targets, colors)),
    title="Held-out metrics: click legend to compare individual digits",
)
for trace in fig.data:
    trace.visible = True if trace.name == "Overall (macro)" else "legendonly"
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(legend_title_text="Click to toggle; double-click to isolate")
fig.show()
fig = px.bar(
    scores_df,
    x="dataset",
    y="accuracy",
    text_auto=".2%",
    title="Held-out classification accuracy",
)
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()


## Error examples

Select up to ten misclassified images per held-out partition with a fixed sampling seed. Captions show true/predicted digits, Kannada numerals, and the assigned cluster. Distances and cluster purity are not softmax probabilities; no fabricated confidence percentage is shown.

In [ ]:
error_frames = []
for name, (features, labels) in evaluation_sets.items():
    predictions, clusters = evaluation_predictions[name]
    wrong = np.flatnonzero(predictions != labels)
    selected = np.random.default_rng(SEED).choice(
        wrong, size=min(10, len(wrong)), replace=False
    )
    if not len(selected):
        print(f"No errors in {name}.")
        continue
    captions = [
        f"True: {labels[i]} ({kannada_digits[labels[i]]})<br>"
        f"Pred: {predictions[i]} ({kannada_digits[predictions[i]]}) · Cluster {clusters[i]}"
        for i in selected
    ]
    digit_gallery(
        features[selected],
        captions,
        f"{name}: ten seeded error examples",
        captions_below=True,
    ).show()
    error_frames.append(
        pd.DataFrame(
            {
                "dataset": name,
                "evaluation_position": selected,
                "true_label": labels[selected],
                "predicted_label": predictions[selected],
                "cluster": clusters[selected],
            }
        )
    )
if error_frames:
    pd.concat(error_frames, ignore_index=True).to_csv(
        output_dir / "error_examples.csv", index=False
    )


## Test predictions and saved artifacts

Predict the unlabeled test set using this experimental model, preserving the original `id,label` template. This is not a full-data refit and does not provide local test accuracy. Save the fitted PCA transform, KMeans model, mapping, split manifest, settings, versions, validation metrics, and prediction CSV under `output/experiment-kmeans-pca-overclustered-mixed/`. On Kaggle the prediction file is written at `/kaggle/working/submission.csv`. No remote execution or competition submission is triggered here.

In [ ]:
x_test = pixel_features(test)
test_predictions, test_clusters = predict_digits(x_test)
submission = submission_template.copy()
submission["label"] = test_predictions
assert submission.columns.tolist() == ["id", "label"]
assert submission["id"].equals(test["id"])
assert submission["label"].between(0, 9).all()
submission_path = (
    Path("/kaggle/working/submission.csv")
    if repo_dir is None and Path("/kaggle/working").is_dir()
    else output_dir / "submission.csv"
)
submission.to_csv(submission_path, index=False)
scores_df.to_csv(output_dir / "evaluation_summary.csv", index=False)
per_digit_df.to_csv(output_dir / "evaluation_by_digit.csv", index=False)
mapping_df.to_csv(output_dir / "cluster_mapping.csv", index=False)
pd.DataFrame(cluster_counts).to_csv(
    output_dir / "cluster_digit_counts.csv", index=False
)
run_manifest = {
    "method": "PCA(90% learning variance) + KMeans(30); TRAIN + Dig learning rows",
    "feature_scaling": "float32 / 255, then PCA retaining at least 90% learning variance without whitening",
    "pca": {
        "config": pca_config,
        "cache_key": pca_cache_key,
        "cache_hit": pca_cache_hit,
        "fit_seconds": pca_fit_seconds,
        "target_explained_variance": variance_target,
        "selected_components": int(pca.n_components_),
        "retained_learning_variance": retained_learning_variance,
        "previous_component_cumulative_variance": previous_learning_variance,
    },
    "distance": "squared Euclidean objective",
    "mapping": "learning-set majority digit per cluster; ties choose smallest digit",
    "numerical_threads": 1,
    "training_sources": training_sources,
    "config": kmeans_config,
    "training_cache_key": training_cache_key,
    "cache_context": cache_context,
    "restart_search": {
        "master_seed": SEED,
        "initializations": N_RESTARTS,
        "seeds": restart_seeds.tolist(),
        "selected_restart": int(best_restart),
        "selected_seed": int(model.random_state),
        "selection": "minimum mixed learning inertia; first run wins exact ties",
    },
    "fit": fit_summary,
    "environment": environment,
    "train_csv_sha256": train_csv_sha256,
    "dig_csv_sha256": dig_csv_sha256,
}
(output_dir / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2) + "\n")
model_cache["run_manifest"] = run_manifest
joblib.dump(model_cache, model_cache_path)
print("Prediction CSV and model diagnostics saved.")
display(submission.head())


## Observed properties of this variance-target PCA model

The **90% cumulative learning-variance rule selected 170 components**. With 169 components the retained variance is **89.9274%**; with 170 it is **90.0169%**. The dimension count is selected from learning images only, not validation labels or accuracy. Full SVD and whitening disabled define the transform.

TRAIN validation accuracy is **89.2583%** with macro F1 **0.891155**. Dig validation accuracy is **58.8867%** with macro F1 **0.567077**. The mapping covers **10 of 10 digits**, including **2 center(s) mapped to digit 7**.

The model uses 170-dimensional PCA distances throughout center fitting, selection, and inference. Original/reconstructed examples and raw-pixel moments explain the representation and assigned members. Explained variance is a representation statistic, not a classification score. Matching PCA and model caches allow reuse without retraining.

Cross-experiment analysis is collected in [report.ipynb](report.ipynb). No PCA/K-means competition score is available, and Saved outputs are exported to HTML; Pages publication follows a merge into main.
